# 07 - Optional CNN-LSTM Experiment

Use normalized PPG windows as time-series input for a compact CNN-LSTM classifier and log results with MLflow.

In [ ]:
import mlflow
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

try:
    import tensorflow as tf
    import mlflow.tensorflow
    from tensorflow.keras import layers, models
    tensorflow_available = True
except Exception as error:
    tensorflow_available = False
    print('TensorFlow is not available on this cluster. Install tensorflow or skip this optional notebook.')
    print(error)

try:
    dbutils.widgets.text('experiment_name', '/Shared/CardioTwin_AI_CNN_LSTM')
    dbutils.widgets.text('epochs', '25')
    experiment_name = dbutils.widgets.get('experiment_name')
    epochs = int(dbutils.widgets.get('epochs'))
except Exception:
    experiment_name = 'CardioTwin_AI_CNN_LSTM'
    epochs = 25

if tensorflow_available:
    clean_pdf = spark.table('cardio_ppg_clean').select('window_id', 'normalized_signal', 'cardiovascular_status').toPandas()
    sequences = [np.asarray(values, dtype=float) for values in clean_pdf['normalized_signal']]
    max_length = min(max(len(values) for values in sequences), 120)

    X = np.zeros((len(sequences), max_length, 1), dtype=np.float32)
    for row_index, values in enumerate(sequences):
        clipped = values[:max_length]
        X[row_index, : len(clipped), 0] = clipped

    label_values = sorted(clean_pdf['cardiovascular_status'].astype(int).unique().tolist())
    label_to_index = {label: index for index, label in enumerate(label_values)}
    y = clean_pdf['cardiovascular_status'].astype(int).map(label_to_index).to_numpy()

    stratify_values = y if min(np.bincount(y)) >= 2 else None
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=stratify_values)
    except ValueError:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

    model = models.Sequential([
        layers.Input(shape=(max_length, 1)),
        layers.Conv1D(filters=24, kernel_size=3, activation='relu'),
        layers.MaxPooling1D(pool_size=2),
        layers.Conv1D(filters=32, kernel_size=3, activation='relu'),
        layers.LSTM(32),
        layers.Dropout(0.20),
        layers.Dense(24, activation='relu'),
        layers.Dense(len(label_values), activation='softmax'),
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )

    mlflow.set_experiment(experiment_name)
    mlflow.tensorflow.autolog(log_models=True)

    with mlflow.start_run(run_name='CNN-LSTM PPG Classifier'):
        mlflow.log_param('max_sequence_length', max_length)
        mlflow.log_param('class_labels', ','.join(str(label) for label in label_values))
        history = model.fit(
            X_train,
            y_train,
            validation_split=0.20,
            epochs=epochs,
            batch_size=8,
            verbose=1,
        )
        test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
        mlflow.log_metric('test_loss', float(test_loss))
        mlflow.log_metric('test_accuracy', float(test_accuracy))

    print(f'CNN-LSTM test accuracy: {test_accuracy:.4f}')
    print('This optional notebook is a prototype. Use a larger real PPG dataset before trusting deep learning results.')
